# W7-X deposition

Neutral beams in DESC's Wendelstein 7-X equilibrium, at production size: the equilibrium sampled once on a grid (`GriddedEquilibrium`), W7-X-like injector boxes with their sources and energy components (`freeofcharge.injectors`), shine-through and birth profiles per source and component, checked against the direct evaluation of DESC.

$\rho$ is the square root of the normalised **toroidal** flux throughout, as in DESC; the profiles and datasets say so (`rho_coordinate`).

**The beam geometry is a stand-in, not W7-X data.** Published (abstracts of Lazerson et al., EPS 2019 and IAEA FEC 2020): hydrogen at 55 keV, about 1.8 MW neutral power per source, two boxes (NI20, NI21) of four sources each, two sources of NI21 in the campaign OP1.2b. The source positions, aiming, divergence, power fractions of the energy components and the armour are IPP's; `W7XInjectors` holds made-up values for them in one parameter set, to be replaced.

## The equilibrium on a grid

DESC's W7-X example (standard configuration, five field periods). Inverting DESC's map point by point (`DESCEquilibrium.rho_at`, through DESC's `map_coordinates`) takes about 15 ms per point, far too slow for beams. `GriddedEquilibrium` samples $\rho^2$ and $\mathbf{B}$ once on a grid over half a field period (stellarator symmetry) and interpolates them, about 0.5 µs per point. For DESC the sampling goes plane by plane (`DESCEquilibrium.rho_and_field_in_planes`: one DESC evaluation per plane on a $(\rho, \theta)$ table, then Newton on its splines), about half a minute at $128 \times 64 \times 128$ nodes. `GriddedEquilibrium.cached` keeps the samples in a netCDF file outside the repository and loads them on the next run.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import plasma_plots as pp
import xarray as xr
from desc.grid import LinearGrid

import freeofcharge as foc
from freeofcharge.desc_equilibrium import DESCEquilibrium
from freeofcharge.injectors import w7x_injectors

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX
warnings.filterwarnings("ignore", category=UserWarning, module="desc")

cache = Path(
    os.environ.get("FREEOFCHARGE_CACHE", Path.home() / ".cache" / "freeofcharge")
)
desc_w7x = DESCEquilibrium.from_example("W7-X")
start = time.perf_counter()
w7x = foc.GriddedEquilibrium.cached(
    cache / "w7x_desc_128_64_128.nc", desc_w7x, (128, 64, 128)
)
print(
    f"grid ready in {time.perf_counter() - start:.1f} s; rho is the root of the {w7x.radial_coordinate} flux"
)

How good is the grid? DESC's own forward map gives the exact $\rho$ and $\mathbf{B}$ at random flux coordinates; the interpolation is compared there. Inside $\rho = 0.9$, $\rho^2$ is accurate to about $2 \times 10^{-4}$ (rms) and $|B|$ to better than $10^{-3}$; the error grows within about two cells of $\rho = 1$, where the samples continue $\rho$ outside with a kink.

In [ ]:
rng = np.random.default_rng(0)
# random flux coordinates on a tensor grid (DESC evaluates those cheaply)
radii = np.sort(np.sqrt(rng.random(60)))
grid = LinearGrid(
    rho=radii,
    theta=np.sort(rng.random(24)) * 2 * np.pi,
    zeta=np.sort(rng.random(6)) * 2 * np.pi / 5,
    NFP=5,
)
data = desc_w7x.equilibrium.compute(["R", "phi", "Z", "|B|"], grid=grid)
r, phi = np.asarray(data["R"]), np.asarray(data["phi"])
points = np.stack([r * np.cos(phi), r * np.sin(phi), np.asarray(data["Z"])], -1)
rho = np.asarray(grid.nodes[:, 0])
error_rho2 = np.abs(np.asarray(w7x.rho_at(points)) ** 2 - rho**2)
error_b = np.abs(
    np.linalg.norm(np.asarray(w7x.field_at(points)), axis=-1) / np.asarray(data["|B|"])
    - 1
)
for lo, hi in ((0.0, 0.9), (0.9, 0.97), (0.97, 1.0)):
    shell = (rho >= lo) & (rho < hi)
    print(
        f"rho in [{lo}, {hi}): rho^2 error rms {np.sqrt(np.mean(error_rho2[shell] ** 2)):.1e}, "
        f"max {error_rho2[shell].max():.1e}; |B| relative error max {error_b[shell].max():.1e}"
    )

## Injector boxes (stand-in geometry)

`w7x_injectors` returns the two boxes. Each aims its axis through a point near the magnetic axis in a triangular plane ($\varphi = 36^\circ$ for NI21, $108^\circ$ for NI20), obliquely (tangency radius 4.3 m), the two in opposite toroidal directions; its four sources converge on a crossover point 1.5 m in front of that point. Every number of the geometry is in `W7XInjectors`; change it with `dataclasses.replace` or describe sources directly with `BeamSource` and `InjectorBox`. Each source has three energy components, the full, half and third energy of the atoms from $\mathrm{H^+}$, $\mathrm{H_2^+}$ and $\mathrm{H_3^+}$, here with the stand-in power shares 50/30/20 %.

In [ ]:
boxes = w7x_injectors()
for box in boxes:
    for s in box.sources:
        print(
            f"{s.name}: at R = {np.hypot(*s.position[:2]):.2f} m, phi = {np.degrees(np.arctan2(s.position[1], s.position[0])):6.1f} deg, "
            f"Z = {s.position[2]:+.2f} m; {s.energy / 1e3:.0f} keV, {s.power / 1e6:.1f} MW, shares {s.power_fractions}"
        )

Seen from above: the last closed surface at $Z = 0$ and the axes of the eight sources, NI21 (orange) co-, NI20 (blue) counter-injecting.

In [ ]:
phi_edge = np.linspace(0.0, 2 * np.pi, 361)
outline = desc_w7x.equilibrium.compute(
    ["R", "Z"], grid=LinearGrid(rho=np.array([1.0]), theta=128, zeta=phi_edge, NFP=1)
)
r_edge = np.asarray(outline["R"]).reshape(len(phi_edge), 128)
z_edge = np.asarray(outline["Z"]).reshape(len(phi_edge), 128)
with pp.figure(figsize=(6, 6)) as fig:
    ax = fig[0]
    # the extent of the plasma in R on Z = 0 (the outermost and innermost points of each section)
    r_out = r_edge.max(axis=1)
    r_in = np.array(
        [
            r_edge[i][np.abs(z_edge[i]) < 0.05].min()
            if np.any(np.abs(z_edge[i]) < 0.05)
            else np.nan
            for i in range(len(phi_edge))
        ]
    )
    for radius in (r_out, r_in):
        ax.plot(
            radius * np.cos(phi_edge),
            radius * np.sin(phi_edge),
            color="black",
            linewidth=1,
        )
    for box, color in zip(boxes, ("tab:blue", "tab:orange"), strict=True):
        for s in box.sources:
            end = s.position + 13.0 * s.direction
            ax.plot(
                [s.position[0], end[0]],
                [s.position[1], end[1]],
                color=color,
                linewidth=1,
            )
    ax.set_aspect("equal")
    ax.set_xlabel("$x$ [m]")
    ax.set_ylabel("$y$ [m]")
    ax.set_title("Stand-in injector geometry, top view")

## Shine-through against density

The two sources of OP1.2b (NI21, S1 and S2) into a hydrogen plasma with parabolic profiles, $T_{e0} = 2$ keV, at central densities from $10^{19}$ to $10^{20}\,\mathrm{m^{-3}}$; 2000 markers per source and component, 1 cm steps.

In [ ]:
op12b = w7x_injectors(sources={"NI21": [1, 2]})
rho_edges = np.linspace(0.0, 1.0, 21)
densities = [1e19, 2e19, 4e19, 7e19, 1e20]
runs = {}
start = time.perf_counter()
for ne0 in densities:
    plasma = foc.Plasma.parabolic(
        ne0, 2e3, ions={"H": (1, 1, 1.0)}, radial_coordinate="toroidal"
    )
    runs[ne0] = foc.deposit_sources(
        op12b, w7x, plasma, markers=2000, seed=1, step=0.01, rho_edges=rho_edges
    )
elapsed = time.perf_counter() - start
markers = len(densities) * 2 * 3 * 2000
print(
    f"{markers} markers in {elapsed:.1f} s ({elapsed / markers * 1e3:.2f} ms per marker)"
)

scan = xr.concat(
    [runs[ne0].to_dataset() for ne0 in densities],
    dim=xr.DataArray(
        densities, dims="ne0", attrs={"long_name": r"$n_{e0}$", "units": "m^-3"}
    ),
)
fraction = scan.shine_through_power.sum("source") / scan.injected_power.sum("source")
fraction.attrs.update(long_name="shine-through fraction")
print(fraction.to_pandas().round(3))
fraction.plasma.plot.profiles(
    x="ne0",
    over="component",
    cmap="tab10",
    title="Shine-through of the two OP1.2b sources",
)

The trend is the published one: the shine-through of the full-energy component falls from over 40 % at the lowest density to below 1 % at the highest, the lower energies are stopped sooner. The published OP1.2b range, up to 60 % for the full energy at low density and about 10 % at high density, comes from an abstract (Lazerson et al., EPS 2019) with no density or profile attached, and the geometry here is a stand-in: the comparison is qualitative only. The oblique tangency radius matters: in this geometry a tangency radius of 3.5 m gives about ten percentage points more shine-through at $10^{19}\,\mathrm{m^{-3}}$ than 4.9 m.

## Birth profiles per source and component

At $n_{e0} = 4 \times 10^{19}\,\mathrm{m^{-3}}$: the power deposited per volume, over $\rho_\mathrm{tor}$, for each energy component (summed over the two sources). The half and third energies deposit further out. The shell volumes come from DESC's flux coordinates (`flux_surfaces`).

In [ ]:
surfaces = w7x.flux_surfaces(rho_edges)
births = runs[4e19].to_dataset(surfaces)
density = births.power_density.sum("source")
density.attrs.update(births.power_density.attrs)
density.plasma.plot.profiles(x="rho", over="component", cmap="tab10")

And per source, all components together: the two sources sit above and below the box axis, so they cross the bean-shaped plasma on slightly different paths.

In [ ]:
per_source = births.power_density.sum("component")
per_source.attrs.update(births.power_density.attrs)
per_source.plasma.plot.profiles(x="rho", over="source", cmap="tab10")

## Against the direct DESC evaluation

The same small beam (four markers of one source's full energy, 2 cm steps) deposited on the grid and on `DESCEquilibrium` itself, which inverts DESC's map at every step. Shine-through and births agree to the interpolation error, while the grid is thousands of times faster.

In [ ]:
plasma = foc.Plasma.parabolic(
    2e19, 2e3, ions={"H": (1, 1, 1.0)}, radial_coordinate="toroidal"
)
beam = op12b[0].sources[0].beams(4, seed=1)["full"]
kwargs = {"step": 0.02, "rho_edges": np.linspace(0.0, 1.0, 11), "seed": 2}
start = time.perf_counter()
direct = foc.deposit(beam, desc_w7x, plasma, **kwargs)
direct_time = time.perf_counter() - start
start = time.perf_counter()
gridded = foc.deposit(beam, w7x, plasma, **kwargs)
gridded_time = time.perf_counter() - start
print(
    f"shine-through: grid {gridded.shine_through_fraction:.4f}, DESC {direct.shine_through_fraction:.4f}"
)
print(
    f"birth profile: largest difference {np.max(np.abs(gridded.birth_rate - direct.birth_rate)) / direct.birth_rate.max():.1e} of its peak"
)
print(
    f"birth positions: largest difference {np.max(np.linalg.norm(np.asarray(gridded.births.positions) - np.asarray(direct.births.positions), axis=1)) * 1e3:.1f} mm"
)
print(f"time: grid {gridded_time:.3f} s, DESC {direct_time:.1f} s")

## Slowing down on the 3D shells

The births of the two OP1.2b sources at $n_{e0} = 4 \times 10^{19}\,\mathrm{m^{-3}}$, all energy components together (`SourceDepositions.total`), slowed down on their shells: drag on the electrons and the ions, pitch-angle scattering, steady state. The local part needs only the shells, so it holds in 3D as in the tokamak; the shell volumes come from the 3D Jacobian of DESC's flux coordinates.

Two parts assume axisymmetry and are guarded on 3D shells: the orbit average (the bounce averages of `freeofcharge.trapping`) is refused, so the pitch is the local one (`orbit_average=False`); and the trapped-electron correction of the driven current (Sauter, Start and Cordey) is a tokamak fit, so without an explicit one the current is NaN, with a warning. Helically trapped ions and 3D orbits are not modelled yet.

In [ ]:
plasma = foc.Plasma.parabolic(
    4e19, 2e3, ions={"H": (1, 1, 1.0)}, radial_coordinate="toroidal"
)
total = runs[4e19].total()
try:
    foc.slow_down(total, plasma, surfaces)
except ValueError as error:
    print(f"orbit average: {error}")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    steady = foc.slow_down(total, plasma, surfaces, orbit_average=False)
print(f"warning: {caught[0].message}")
print(
    f"absorbed {total.absorbed_power / 1e6:.3f} MW: electrons {steady.electron_heating.sum() / 1e6:.3f} MW, "
    f"ions {steady.ion_heating.sum() / 1e6:.3f} MW, power balance {steady.power_balance:+.1e}"
)
print(f"fast-ion energy {steady.stored_energy / 1e3:.1f} kJ")

The heating per volume of the electrons and the ions, and the fast-ion density and pressure, over $\rho_\mathrm{tor}$. At 2 keV the critical energy of hydrogen in hydrogen is about 30 keV: the full energy (55 keV) heats mostly the electrons, the half and third energies (27 and 18 keV) mostly the ions, about half and half in all.

In [ ]:
profiles = steady.to_dataset(surfaces)
profiles.heating_density.plasma.plot.profiles(x="rho", over="species").show()
profiles.fast_ion_density.plasma.plot.lineout(x="rho").show()
profiles.fast_ion_pressure.plasma.plot.lineout(x="rho")

Volume integrals in physical space: random points in the box around the plasma, counted by shell of the grid's $\rho$ (`monte_carlo_volumes`), give the shell volumes without flux coordinates. They agree with the Jacobian's to about 0.4 % (the grid's last closed surface is off by a fraction of a cell), and the deposited power density integrated over them gives back the absorbed power to the same accuracy.

In [ ]:
volume, error = foc.monte_carlo_volumes(w7x, rho_edges, samples=4_000_000, seed=1)
print(
    f"plasma volume: Jacobian {surfaces.volume.sum():.2f} m^3, Monte Carlo {volume.sum():.2f} +- {np.sqrt(np.sum(error**2)):.2f} m^3"
)
print(
    f"largest shell difference: {np.max(np.abs(volume - surfaces.volume) / error):.1f} standard errors"
)
deposited = np.sum(total.power_density(surfaces) * volume)
heated = np.sum((profiles.heating_density.sum("species") * volume).values)
print(
    f"absorbed {total.absorbed_power / 1e6:.3f} MW; volume integrals: deposited {deposited / 1e6:.3f} MW, heating {heated / 1e6:.3f} MW"
)